# Topological kinematics of collective motion: the figures, explained

This notebook rebuilds every figure of the paper, in the order of the paper, and explains how to read it. Figure
numbers are those of the paper, `manuscript/paper.pdf`.
Nothing here recomputes distances: the heavy computations are done by

* `run_pipeline.py` (stages 1-3: fields, extended persistence, sliced-Wasserstein distances),
* `run_sweep.py` (clustering: resolution grid, extended grid, standard SNF, ablation, taxonomy),
* `analysis/run_all.sh` (every clustering experiment in order), `analysis/attribution.py` (back-propagation),
  `analysis/merge_analysis.py`, `analysis/invariance_check.py`.

All results live in `results/epuck/paper/`; figures go to `figures/`; the numbers quoted in the paper are written
to `manuscript/numbers.tex` by `analysis/make_numbers.py`.

The figures of Sections 3 and 4 come in two kinds: small **synthetic** examples (`analysis/illustrations.py`),
which show each idea on a toy swarm with the same closed forms as the pipeline, and the same steps on the
**robot data** (`analysis/make_figures.py`).

In [ ]:
import sys, pickle
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name in ("analysis", "notebooks") else Path.cwd()
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "src"), str(ROOT)]
import numpy as np
from IPython.display import Image, display
import make_figures as M
import illustrations as I
import figlib as L
FIG = ROOT / "figures"
def show(name, width=900):
    display(Image(filename=str(FIG / f"{name}.png"), width=width))

## 1. Overview (Figure 1)

The boxes of Figure 1 hold one small picture per step, all from the robot data: the tracks of the Shelter
exemplar, its five fields, the diagram of its density, one distance matrix and the consensus of Top-1 (runs
ordered by the discovered clusters), and the five exemplars.

In [ ]:
M.fig_thumbs()
for name in ("thumb_tracks", "thumb_fields", "thumb_diagram", "thumb_distance", "thumb_consensus", "thumb_exemplars"):
    print(name, (FIG / f"{name}.pdf").exists())

## 2. From agents to fields, on toy swarms (Figures 2-5)

**The Gaussian footprint (Figure 2).** Each agent becomes a smooth bump of width $\sigma$. Two bumps stay
separate when their centres are more than $2.35\,\sigma$ apart; below, they merge. The width is therefore a lens:
small $\sigma$ resolves every agent, large $\sigma$ shows the groups they form.

**The three moments (Figure 3).** $M_0$ says where the agents are, $M_1$ their net flux, $\operatorname{tr} M_2$
their agitation. A group at rest has no flux; two opposite streams cancel in $M_1$ but not in $M_2$.

**The potentials (Figure 4).** Gathering creates a sink (maximum of $\varphi$), dispersing a source (minimum of
$\varphi$), milling a rotation centre (extremum of $\psi$).

**The coupling (Figure 5).** Two swarms with the same ingredients, one sink and one vortex, arranged differently:
when the sink and the rotation centre coincide, $J$ nearly vanishes; when they lie apart, $J$ lights up between them.

In [ ]:
I.fig_footprint(); show("fig_footprint")
I.fig_moments(); show("fig_moments")
I.fig_flows(); show("fig_flows")
I.fig_potentials(); show("fig_potentials")

## 3. The ladder of scales and persistence (Figure 6)

**The ladder (Figure 6).** The density and the vorticity potential of the Shelter exemplar at the four scales:
at the finest scale the fields resolve individual paths and the walls of the shelter, at the coarsest the
organisation of the whole group. The five fields at the four scales are in the appendix.

**Persistence and the sliced Wasserstein distance (Figure 7).** Sublevel sets grow with the level; components are
born at minima and die when they merge; each birth-death pair is a point of the diagram. Two diagrams are compared
by projecting their points (augmented with the diagonal projections of the other) on 25 directions, sorting, and
matching in order.

In [ ]:
I.fig_persistence(); show("fig_persistence")

## 4. The method on the robot data (Figures 7-8)

**One recording (Figure 7)**, the Shelter exemplar: (a) the tracks; (b) its five fields at $\sigma_3$; (c) the
**grid-free domain**: robot positions kept as seeds (grey), seeds near critical points of the fields (orange) and seeds on the
coupling skeleton $J=0$ (blue), joined by the alpha complex; (d) the extended-persistence diagrams of the
density (square-root axes; Ext$_1$ is empty, since this domain has no hole).

**The collection (Figure 8)**, the 250 runs of Top-1: (a) the 20 rank-normalised similarity matrices (5 fields x 4
scales); (b) the four balanced views $W_m$ (equal weight per moment); (c) the consensus $S^\star$ after one
SNF across scales, ordered by the discovered clusters, stars at the exemplars; (d) the five exemplar recordings.

In [ ]:
M.fig_pipeline_one(); show("fig_pipeline_one")
M.fig_pipeline_all(); show("fig_pipeline_all")

## 5. The taxonomy in one table (Table 2)

For each dataset Top-$n$ ($n$ best controllers of each of the five missions, $250n$ runs), the method sweeps
99 resolutions $(k_{nn}, t)$, keeps the **plateau** (the settings where the number of clusters takes its most
frequent value $C^\star$), and builds the taxonomy as the **consensus** of the plateau partitions.

* *stability*: mean ARI between each plateau partition and the consensus (label-free);
* *purity*: fraction of runs in a cluster dominated by their mission; *misassigned*: runs that are not;
* *homogeneity* = 1 when no cluster mixes missions; *completeness* = 1 when no mission is split.

Pooled datasets keep a high homogeneity and lose completeness: missions are **split** into controller-level
sub-strategies, not **mixed**.

In [ ]:
T = pickle.load(open(ROOT / "results/epuck/paper/taxonomy.pkl", "rb"))
print(f"{'set':>6} {'C*':>3} {'plateau':>8} {'stab':>6} {'purity':>7} {'miss':>5} {'ARI':>6} {'hom':>6} {'comp':>6}")
for top, t in T.items():
    print(f"Top-{top:<2} {t['C']:>3} {t['n_plateau']:>5}/99 {t['stability']:6.3f} {t['purity']:7.3f} {t['misassigned']:5d} "
          f"{t['ari']:6.3f} {t['homogeneity']:6.3f} {t['completeness']:6.3f}")

## 6. What the five fields see (Figure 10)

One exemplar per discovered behaviour, at $\sigma_3$. Sequential colours for the density $\rho$ and the kinetic
intensity $K$; diverging colours (blue negative, orange positive) for the divergence potential $\varphi$, the
vorticity potential $\psi$ and the Jacobi coupling $J$.

* **D-Gate**: corridor and gate in $\rho$, $K$; $\varphi$ splits the arena north (source) / south (sink).
* **Homing**: one dense spot in $\rho$, but $K$ spread over the arena (the robots cross it before gathering).
* **Shelter**: the shelter and the lit wall; opposite rotations around the shelter in $\psi$.
* **XOR**: nearly all the robots on one of its two black areas.

In [ ]:
M.fig_fields(); show("fig_fields")

## 7. Resolution maps (Figure 9 for Top-1 and Top-10, Figure 18 for the ten datasets)

Each cell is one setting $(k_{nn}, t)$: the number is the count $C$ of clusters, the colour is the purity, and the
white contour outlines the **plateau**. Fine resolution (bottom-left) splits missions into pure pieces; coarse
resolution (top-right) merges neighbouring missions.

In [ ]:
M.fig_resolution_compact(); show("fig_resolution_compact")
M.fig_resolution(); show("fig_resolution_map", 700)

## 8. The hierarchy of missions and what explains it (Figures 11-12)

Extended grid on Top-1. No setting of the main grid joins two missions. For every neighbourhood large enough to merge
within the extended grid, the first merge is **D-Gate + Foraging** (one mission absorbed whole: the merged cluster holds
exactly the runs of the two missions), then **Shelter + XOR**; **Homing** stays apart over the whole extended grid. The
separation heatmap below explains why:

* D-Gate / Foraging: the closest pair in $\rho$, $K$, $\varphi$ and $J$; separated only by $\psi$ (how the traffic turns);
* Shelter / XOR: best separated by the coupling $J$, which records where the flow converges relative to where it turns;
* Homing: mixed signature, like the aggregating missions in $\rho$ and $J$, like the moving ones in $K$.

In [ ]:
M.fig_fusion_sequence(); show("fig_fusion_sequence")
M.fig_pair_separation(); show("fig_pair_separation")

## 9. Which fields matter (Figure 13)

Purity and ARI of eleven field combinations over the ten datasets and 99 settings (990 values each), each combination
fused with the same rule (equal weight per moment present).

In [ ]:
M.fig_ablation(); show("fig_ablation")

## 10. Explaining a distance: back-propagation to the robots (Figure 14)

The distance between the D-Gate and the Foraging exemplars, on the vorticity potential at $\sigma_3$, is
differentiated with respect to the field at the seeds (left), then to every robot position (middle), and repeated
over the 50 runs of the D-Gate controller (right). Top: extended persistence, whose gradient lands on critical
points; bottom: the distribution of the field values, whose gradient lands mostly on ordinary points. The full
computation, with every check against the pipeline, is in `07_attribution.ipynb`.

In [ ]:
M.fig_attribution(); show("fig_attribution", 800)

## 11. Appendix: the five fields at four scales, and why the ladder is capped (Figures 15-16)

In [ ]:
M.fig_scales(); show("fig_scales", 700)
M.fig_uncapped(); show("figA_uncapped")

## 12. Appendix: the XOR Aggregation runs placed with Shelter (Figure 19)

With percentile ranks alone, the taxonomies of Top-3 and Top-4 place some XOR Aggregation runs in the Shelter cluster;
the mutual ranks of the method keep almost all of them with their mission. Their robots end on the XOR black areas,
like the other XOR runs, but keep moving almost as long as in Shelter: averaged over the recording, they move like
Shelter. Computed by `analysis/merge_analysis.py`.

In [ ]:
M.fig_merge(); show("figA_merge")

## 13. Regenerating the numbers of the paper

`analysis/make_numbers.py` writes `manuscript/numbers.tex` (every number in the text) and the tables in
`manuscript/tables/`. Then compile `manuscript/paper.tex`.

In [ ]:
import subprocess
print(subprocess.run([sys.executable, str(ROOT / "analysis/make_numbers.py")], capture_output=True, text=True).stdout)